# Comprehensive Model Evaluation — iteris

Cross-cutting evaluation of every model this project has produced: U-Net baselines
(**LiteUNet**, **AttentionResUNet**), the **BRISC tumor-type classifier**, and DRL
refiners (**DuelingDDQN** — discrete contour-sector actions, **TD3** — continuous
contour-sector actions) across both datasets (**CAMUS**: LV_endo / LV_epi / LA,
**BRISC**: glioma / meningioma / pituitary / tumor).

This notebook does **not** hardcode any results, and does **not** import `iteris`/
`torch`/`monai` for inference — it only reads whatever run-output *files* you drop
into `results/` (locally) or attach as Kaggle input datasets, and computes every
chart/table/verdict directly from them. If a section has no matching files it
prints what it's missing instead of fabricating numbers.

**Charting**: [Plotly](https://plotly.com/python/) (px/go) throughout instead of
static matplotlib -- the one exception is Section 9's image gallery, which just
displays pre-rendered PNG files the training pipeline already produced (there's
no "chart" to modernize there, it's a raster viewer). Every chart is rendered as
a **static PNG** rather than Plotly's default interactive HTML/JS output: GitHub's
notebook viewer, nbviewer, and Kaggle's plain "Log" console don't execute that JS,
so the default renderer would leave every chart cell looking empty in exactly
those places -- a static image has no such dependency and shows up everywhere.
The setup cell pins a compatible export stack (`plotly<6` + `kaleido==0.2.1` --
Kaggle preinstalls a mismatched `plotly 5.24` + `kaleido 1.3` pair that silently
breaks static export) and probes it once up front; if static export genuinely
can't be made to work (e.g. pip offline), charts fall back to Plotly's interactive
renderer with a single printed warning (visible in a live Jupyter/Kaggle session,
not on GitHub).

**Framing**: discrete (DuelingDDQN) vs continuous (TD3) is compared two ways --
head-to-head on **absolute deployed Dice** (Section 4, the primary framing: which
action space produces the better mask, full stop, independent of how strong its
particular baseline was) and, separately, on **delta vs U-Net baseline** (Section
5, a different question: how much *learning* happened). Neither framing subsumes
the other -- a small delta can still mean a high absolute Dice if the baseline was
already strong, and vice versa.

## Where it looks for files

- **Locally**: `results/` (or `../results` / `../../results`, whichever exists),
  searched recursively — subfolders are fine.
- **On Kaggle**: every dataset attached under `/kaggle/input/` is searched
  recursively too, so you can attach all your output datasets to one notebook and
  just run it. Outputs (figures, exported tables) go to
  `/kaggle/working/evaluation_outputs/` there, since `/kaggle/input` is read-only.

Every file is identified by its **content shape** (or, for CSVs/PNGs/checkpoints
that carry no metadata of their own, by matching them to a same-folder sibling
JSON with the same filename stem) — never by which folder you put it in.

## What's recognized automatically

| File | Produced by | Identified by |
|---|---|---|
| `<dataset><suffix>_summary.json` | `iteris.evaluation.save_summary_json` | `test_dice` key |
| `<dataset><suffix>_test_scores.csv` | `iteris.evaluation.evaluate_test_set` | per-patient `dice_<class>` columns |
| `<dataset><suffix>_learning_curves.png` | `iteris.visualization.plot_learning_curves` | filename |
| `<dataset><suffix>_qualitative.png` | `iteris.visualization.plot_qualitative_grid` | filename |
| `<dataset>_<agent>_c<class>_reeval_test_results.json` | `iteris.drl_reeval.reeval_checkpoint` | `init_dice_mean`+`final_dice_mean` keys |
| `..._reeval_comparison.png` / `..._reeval_behaviour.png` | same (`plot_comparison`/`plot_behaviour`) | filename |
| `brisc_tumor_classifier_summary.json` | `iteris.classifier.save_classifier_summary_json` | `accuracy`+`per_class` keys |
| `brisc_tumor_classifier_learning_curves.png` / `..._confusion_matrix.png` | classifier training/eval notebook | filename |
| `*.pt` checkpoints | any training run | optional, `torch`-only, scalar fields only |

Optional, for real significance testing beyond the U-Net per-patient CSVs: a
per-sample CSV per **DRL** run with `init_dice`/`final_dice` columns (nothing in
the current pipeline exports this yet — see Section 11).

## Sections
1. Ingestion
2. Master tidy comparison table
3. U-Net baseline landscape — grouped bars, per-metric small multiples, violin + ECDF of per-patient Dice
4. Discrete vs continuous, on absolute Dice (head-to-head, not vs U-Net)
5. Discrete vs continuous, on delta vs U-Net baseline (supporting view -- learning behaviour)
6. Per-class / per-dataset learning behaviour (delta heatmap + absolute-Dice heatmap)
7. Multi-metric radar comparison (all models, all metrics, per dataset)
8. Most-improved DRL agent ranking -- by delta AND by absolute Dice
9. Qualitative visual gallery (comparison / behaviour / learning-curve / prediction images)
10. BRISC tumor-type classifier evaluation — table, bar chart, radar chart, gauge, confusion matrix
11. Checkpoint / training provenance (optional, needs `torch`)
12. Statistical significance (Wilcoxon signed-rank + Bonferroni) + DRL metric correlation heatmap
13. Export & auto-generated summary


In [ ]:
import json, os, sys, subprocess
from pathlib import Path
import concurrent.futures

import numpy as np
import pandas as pd
from scipy import stats

# ── Ensure a MUTUALLY-COMPATIBLE plotly + kaleido pair for static PNG export ───
# Why this is needed: Kaggle preinstalls plotly 5.24 + kaleido 1.3, which are
# incompatible -- kaleido 1.x requires plotly>=6.1.1, while plotly 5.x needs
# kaleido's OLD 0.2.x API. The mismatch makes every fig.to_image() fail with a
# misleading "requires the kaleido package" error, so charts silently vanish.
# And static PNGs matter: plotly's DEFAULT renderer embeds interactive HTML+JS
# that only runs in a LIVE kernel; GitHub's notebook viewer, nbviewer, and
# Kaggle's plain "Log" console don't execute that JS, so the default renderer
# leaves every chart cell blank in exactly those places. Pin the classic,
# self-contained combo: kaleido 0.2.1 bundles its own Chromium (no separate
# Chrome download that could stall/hang) and pairs with the plotly 5.x already
# present. No-op on an env that already satisfies it; harmless if pip is offline
# (the probe below then just flips to the interactive fallback with one message).
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'plotly>=5.24,<6', 'kaleido==0.2.1'], check=False)
except Exception as _e:
    print(f'[config] could not pin plotly/kaleido ({_e}); continuing with whatever is installed.')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'
import matplotlib.image as mpimg   # raster viewer only (Section 9's PNG gallery), not for charting
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display as ipy_display

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

_kaleido_executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
_kaleido_broken = False

# ── One-time probe: is static PNG export actually functional? ─────────────────
# Do the expensive check ONCE here (a tiny to_image call, bounded by a timeout in
# case a broken kaleido/Chrome hangs instead of erroring) so a mismatch produces a
# SINGLE clear message rather than the same multi-line error repeated per chart.
try:
    import kaleido
    HAVE_KALEIDO = True
except Exception:
    HAVE_KALEIDO = False

if HAVE_KALEIDO:
    try:
        _probe = _kaleido_executor.submit(go.Figure().to_image, format='png')
        _probe.result(timeout=30)
    except Exception as e:
        HAVE_KALEIDO = False
        _msg = str(e).strip().replace(chr(10), ' ')[:160]
        print(f'[config] static PNG export is NON-functional ({_msg}). Charts will use plotly\'s '
              f'interactive renderer, which does NOT display on GitHub/nbviewer (only in a live '
              f'Jupyter/Kaggle session). Fix: ensure plotly<6 with kaleido==0.2.1, or plotly>=6.1.1 with kaleido>=1.')
else:
    print('[config] kaleido not importable -- charts will use the interactive renderer '
          '(not visible on GitHub/nbviewer, only in a live session).')

try:
    import torch
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False

def render(fig, name):
    """Display `fig` as a static PNG (visible on GitHub/nbviewer/Kaggle's Log, none
    of which run the JS plotly's default interactive renderer needs) and save a copy
    to FIG_DIR. If static export was probed as non-functional at setup (HAVE_KALEIDO
    False), or fails/times out mid-run, fall back to the interactive renderer -- still
    viewable in a live session, just not on GitHub -- and stop retrying so the log
    isn't flooded with the same error once per chart.
    """
    global _kaleido_broken
    if HAVE_KALEIDO and not _kaleido_broken:
        future = _kaleido_executor.submit(fig.to_image, format='png', scale=2)
        try:
            png_bytes = future.result(timeout=30)
            ipy_display(IPyImage(data=png_bytes))
            (FIG_DIR / name).write_bytes(png_bytes)
            return
        except concurrent.futures.TimeoutError:
            _kaleido_broken = True
            print('  [note] static render timed out -- interactive fallback for the rest of this run.')
        except Exception as e:
            _kaleido_broken = True
            print(f'  [note] static render failed ({str(e).strip()[:100]}) -- interactive fallback for the rest of this run.')
    fig.show()


In [ ]:
# Configuration -- where to search for files, where to write outputs.
_local_candidates = [Path('../../results'), Path('results'), Path('../results')]
LOCAL_RESULTS = next((p for p in _local_candidates if p.exists()), None)
KAGGLE_INPUT = Path('/kaggle/input')

SEARCH_ROOTS = []
if LOCAL_RESULTS is not None:
    SEARCH_ROOTS.append(LOCAL_RESULTS)
if KAGGLE_INPUT.exists():
    SEARCH_ROOTS.append(KAGGLE_INPUT)
if not SEARCH_ROOTS:
    SEARCH_ROOTS = [_local_candidates[1]]  # 'results' -- may not exist yet, that's fine

if Path('/kaggle/working').exists():
    OUT_DIR = Path('/kaggle/working/evaluation_outputs')
elif LOCAL_RESULTS is not None:
    OUT_DIR = LOCAL_RESULTS
else:
    OUT_DIR = Path('results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('[config] searching for result files under:')
for r in SEARCH_ROOTS:
    print(f'    - {r.resolve()}')
print(f'[config] writing figures/exports to: {OUT_DIR.resolve()}')
print(f'[config] torch available: {HAVE_TORCH}   kaleido (static chart rendering) available: {HAVE_KALEIDO}')

def find_files(exts):
    # Exclude only this notebook's own exported figures/table (NOT all of OUT_DIR --
    # locally OUT_DIR often *is* a search root itself, so excluding the whole tree
    # would hide the real input files too).
    fig_dir_resolved = FIG_DIR.resolve()
    out = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob('*'):
            if not p.is_file() or p.suffix.lower() not in exts:
                continue
            resolved = p.resolve()
            if fig_dir_resolved in resolved.parents:
                continue
            if resolved.name == 'master_comparison.csv':
                continue
            out.append(p)
    return sorted(set(out))

# agent_type (as stored in cfg / saved JSON) -> action space + display name
AGENT_META = {
    'TD3':      dict(action_space='continuous', display='TD3'),
    'DUELING':  dict(action_space='discrete',   display='DuelingDDQN'),
    'DDPG':     dict(action_space='continuous', display='DDPG (archived global)'),
    'DQN':      dict(action_space='discrete',   display='DQN (archived global)'),
}
# internal cfg['model'] key -> display name (matches iteris.evaluation._model_names)
UNET_META = {
    'lite_unet':        'LiteUNet',
    'attn_resunet':     'AttentionResUNet',
    'LiteUNet':         'LiteUNet',
    'AttentionResUNet': 'AttentionResUNet',
}
DATASET_CLASS_ORDER = {
    'CAMUS': ['LV_endo', 'LV_epi', 'LA'],
    'BRISC': ['glioma', 'meningioma', 'pituitary', 'tumor'],
}

# Filename-stem matching so CSVs/PNGs/checkpoints (which carry no metadata of
# their own) can inherit dataset/model/class/agent from a same-folder sibling
# JSON, purely from filename structure -- never by directory naming.
JSON_SUFFIXES = ['_summary.json', '_reeval_test_results.json']

def strip_suffix(name, suffixes):
    for s in suffixes:
        if name.endswith(s):
            return name[:-len(s)]
    return None

def sibling_meta(path, own_suffixes, json_meta):
    """Find a same-folder JSON whose stem matches this file's stem, return its meta dict."""
    stem = strip_suffix(path.name, own_suffixes)
    if stem is None:
        return None
    for jpath_str, meta in json_meta.items():
        jpath = Path(jpath_str)
        if jpath.parent != path.parent:
            continue
        if strip_suffix(jpath.name, JSON_SUFFIXES) == stem:
            return meta
    return None


## 1. Ingestion\n\nWalk every file under the search roots, sniff its shape/name, and route it. Anything unrecognized is reported, not silently dropped.

In [ ]:
def _load_json(path):
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f'  [skip] {path}: could not parse ({e})')
        return None

def parse_unet_summary(d, path):
    rows = []
    dataset = d.get('dataset', '?')
    model = UNET_META.get(d.get('model', '?'), d.get('model', '?'))
    for cls, dice in d.get('test_dice', {}).items():
        rows.append(dict(
            source=str(path), dataset=dataset, model=model, model_family='UNet',
            action_space='n/a', class_name=cls, dice=dice,
            iou=d.get('test_iou', {}).get(cls), biou=d.get('test_biou', {}).get(cls),
            precision=d.get('test_precision', {}).get(cls), sensitivity=d.get('test_sensitivity', {}).get(cls),
            hd95=d.get('test_hd95', {}).get(cls), hd95geo=d.get('test_hd95geo_px', {}).get(cls),
            msd=d.get('test_msd_px', {}).get(cls), best_val_dice=d.get('best_val_dice'),
            label_frac=d.get('label_frac'),
        ))
    return rows, dict(dataset=dataset, model=model, kind='unet')

def parse_drl_summary(d, path):
    agent_type = str(d.get('agent_type', '?')).upper()
    meta = AGENT_META.get(agent_type, dict(action_space='unknown', display=agent_type))
    row = dict(
        source=str(path), dataset=d.get('dataset', '?'), class_name=d.get('class_name', '?'),
        target_class=d.get('target_class'), model=meta['display'], model_family='DRL',
        action_space=meta['action_space'], agent_type=agent_type,
        init_dice=d.get('init_dice_mean'), final_dice=d.get('final_dice_mean'),
        best_dice_ceiling=d.get('best_dice_mean'), value_floored_dice=d.get('value_floored_dice_mean'),
        delta_dice=d.get('delta_dice_mean'), value_floored_delta=d.get('value_floored_delta_mean'),
        final_hd95=d.get('final_hd95_mean'), init_hd95=d.get('init_hd95_mean'),
        final_iou=d.get('final_iou_mean'), init_iou=d.get('init_iou_mean'), delta_iou=d.get('delta_iou_mean'),
        final_biou=d.get('final_biou_mean'), init_biou=d.get('init_biou_mean'), delta_biou=d.get('delta_biou_mean'),
        final_precision=d.get('final_precision_mean'), final_sensitivity=d.get('final_sensitivity_mean'),
        final_msd=d.get('final_msd_mean'), init_msd=d.get('init_msd_mean'),
        n_refinable=d.get('n_refinable'), n_total=d.get('n_total'), refinable_frac=d.get('refinable_frac'),
        value_floored_dice_refinable=d.get('value_floored_dice_refinable_mean'),
        value_floored_delta_refinable=d.get('value_floored_delta_refinable_mean'),
        routed_dice=d.get('routed_dice_mean'), routed_delta=d.get('routed_delta_mean'),
        reeval=d.get('reeval', False), source_checkpoint=d.get('source_checkpoint'),
    )
    meta = dict(dataset=row['dataset'], class_name=row['class_name'], agent_type=agent_type,
                model=meta['display'], kind='drl')
    return [row], meta

def parse_classifier_summary(d, path):
    return dict(source=str(path), best_val_acc=d.get('best_val_acc'),
                accuracy=d.get('test_metrics', {}).get('accuracy'),
                n_test=d.get('test_metrics', {}).get('n_test'),
                per_class=d.get('test_metrics', {}).get('per_class', {}),
                epochs_trained=d.get('epochs_trained'), class_names=d.get('class_names'))

unet_rows, drl_rows, classifier_summaries, unrecognized_json = [], [], [], []
json_meta = {}   # str(path) -> dict(dataset=, model=, class_name=, agent_type=, kind=)
for path in find_files({'.json'}):
    d = _load_json(path)
    if d is None:
        continue
    if isinstance(d, dict) and 'test_dice' in d:
        rows, meta = parse_unet_summary(d, path)
        unet_rows += rows
        json_meta[str(path)] = meta
    elif isinstance(d, dict) and {'init_dice_mean', 'final_dice_mean'} <= d.keys():
        rows, meta = parse_drl_summary(d, path)
        drl_rows += rows
        json_meta[str(path)] = meta
    elif isinstance(d, dict) and {'accuracy', 'per_class'} <= d.get('test_metrics', {}).keys():
        classifier_summaries.append(parse_classifier_summary(d, path))
    else:
        unrecognized_json.append(path)

unet_df = pd.DataFrame(unet_rows)
drl_df = pd.DataFrame(drl_rows)
if not drl_df.empty:
    drl_df['deploy_dice'] = drl_df['value_floored_dice'].fillna(drl_df['final_dice'])
    drl_df['deploy_delta'] = drl_df['value_floored_delta'].fillna(drl_df['delta_dice'])
    drl_df['drift'] = drl_df['best_dice_ceiling'] - drl_df['final_dice']

print(f'[ingestion] U-Net summary rows: {len(unet_df)}  (files: {unet_df["source"].nunique() if len(unet_df) else 0})')
print(f'[ingestion] DRL summary rows:   {len(drl_df)}  (files: {drl_df["source"].nunique() if len(drl_df) else 0})')
print(f'[ingestion] classifier summaries: {len(classifier_summaries)}')
if unrecognized_json:
    print(f'[ingestion] {len(unrecognized_json)} JSON file(s) matched no known shape, skipped:')
    for p in unrecognized_json:
        print(f'    - {p}')
if unet_df.empty and drl_df.empty and not classifier_summaries:
    print(f"""
No result files found under any of: {[str(r) for r in SEARCH_ROOTS]}.
Drop your *_summary.json / *_reeval_test_results.json / brisc_tumor_classifier_summary.json
files anywhere under a searched root (subfolders OK, or attach as Kaggle datasets), then re-run.
""")


In [ ]:
# ── CSVs: U-Net per-patient test_scores (real paired data!) + generic DRL per-sample ──
unet_csv_frames = []       # list of dict(path, dataset, model, df)
generic_paired_frames = {} # path -> df, for the init_dice/final_dice shape (DRL, if ever exported)
other_csv = []

for path in find_files({'.csv'}):
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f'  [skip] {path}: {e}')
        continue
    if 'patient' in df.columns and any(c.startswith('dice_') for c in df.columns):
        meta = sibling_meta(path, ['_test_scores.csv'], json_meta) or {}
        unet_csv_frames.append(dict(path=str(path), dataset=meta.get('dataset', '?'),
                                     model=meta.get('model', '?'), df=df))
    elif {'init_dice', 'final_dice'} <= set(df.columns):
        generic_paired_frames[str(path)] = df
    else:
        other_csv.append(path)

print(f'[ingestion] U-Net per-patient CSVs: {len(unet_csv_frames)}')
for f in unet_csv_frames:
    tag = f'{f["dataset"]}/{f["model"]}' if f['model'] != '?' else '(no sibling summary.json found to tag model)'
    print(f'    - {f["path"]}  [{tag}]  n={len(f["df"])}')
print(f'[ingestion] generic paired (init_dice/final_dice) CSVs: {len(generic_paired_frames)}')
for k in generic_paired_frames:
    print(f'    - {k}')
if other_csv:
    print(f'[ingestion] {len(other_csv)} CSV(s) matched no known shape, skipped:')
    for p in other_csv:
        print(f'    - {p}')


In [ ]:
# ── PNGs: classify by filename, tag via sibling JSON where applicable ──
png_gallery = {'comparison': [], 'behaviour': [], 'learning_curves': [],
               'qualitative': [], 'classifier_confusion': [], 'classifier_curves': [], 'other': []}

for path in find_files({'.png'}):
    name = path.name.lower()
    if 'classifier' in name and 'confusion' in name:
        png_gallery['classifier_confusion'].append(dict(path=path, caption=path.stem))
    elif 'classifier' in name:
        png_gallery['classifier_curves'].append(dict(path=path, caption=path.stem))
    elif name.endswith('_reeval_comparison.png'):
        meta = sibling_meta(path, ['_reeval_comparison.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')}/{meta.get('class_name','?')} — {meta.get('model','?')} (comparison)"
        png_gallery['comparison'].append(dict(path=path, caption=cap))
    elif name.endswith('_reeval_behaviour.png'):
        meta = sibling_meta(path, ['_reeval_behaviour.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')}/{meta.get('class_name','?')} — {meta.get('model','?')} (behaviour)"
        png_gallery['behaviour'].append(dict(path=path, caption=cap))
    elif name.endswith('_learning_curves.png'):
        meta = sibling_meta(path, ['_learning_curves.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')} — {meta.get('model','?')} (learning curves)" if meta else path.stem
        png_gallery['learning_curves'].append(dict(path=path, caption=cap))
    elif name.endswith('_qualitative.png'):
        meta = sibling_meta(path, ['_qualitative.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')} — {meta.get('model','?')} (qualitative)" if meta else path.stem
        png_gallery['qualitative'].append(dict(path=path, caption=cap))
    else:
        png_gallery['other'].append(dict(path=path, caption=path.stem))

for k, v in png_gallery.items():
    print(f'[ingestion] {k}: {len(v)} image(s)')


In [ ]:
# ── Checkpoints: optional, torch-only, scalar metadata (never rebuilds a model) ──
ckpt_rows = []
if HAVE_TORCH:
    for path in find_files({'.pt', '.pth'}):
        try:
            ck = torch.load(path, map_location='cpu', weights_only=False)
        except Exception as e:
            print(f'  [skip] {path}: could not load ({e})')
            continue
        if not isinstance(ck, dict):
            continue
        ckpt_rows.append(dict(
            path=str(path), filename=path.name,
            step=ck.get('step'), epoch=ck.get('epoch'),
            best_dice=ck.get('best_dice'), val_acc=ck.get('val_acc'),
            class_names=ck.get('class_names'),
        ))
    print(f'[ingestion] checkpoints with readable metadata: {len(ckpt_rows)}')
else:
    print('[ingestion] torch not importable -- skipping checkpoint metadata (Section 11 will note this).')
ckpt_df = pd.DataFrame(ckpt_rows)


## 2. Master tidy comparison table\n\nOne row per (dataset, class, model), Dice as the common column, tagged by `model_family` (UNet / DRL) and `action_space` (n/a / discrete / continuous).

In [ ]:
master_rows = []
if not unet_df.empty:
    for _, r in unet_df.iterrows():
        master_rows.append(dict(dataset=r['dataset'], class_name=r['class_name'],
                                 model=r['model'], model_family='UNet', action_space='n/a',
                                 dice=r['dice'], hd95=r['hd95'], iou=r['iou']))
if not drl_df.empty:
    for _, r in drl_df.iterrows():
        master_rows.append(dict(dataset=r['dataset'], class_name=r['class_name'],
                                 model=r['model'], model_family='DRL', action_space=r['action_space'],
                                 dice=r['deploy_dice'], hd95=r['final_hd95'], iou=r['final_iou']))

master_df = pd.DataFrame(master_rows)
if not master_df.empty:
    master_df = master_df.sort_values(['dataset', 'class_name', 'model_family', 'model']).reset_index(drop=True)
master_df


## 3. U-Net baseline landscape

`LiteUNet` is the deliberately weak baseline the DRL agents refine (chosen because it has
headroom); `AttentionResUNet` is the strong upper-bound competitor. The gap between them,
per class, is the ceiling a DRL agent could theoretically reach.

In [ ]:
if unet_df.empty:
    print('No U-Net summaries loaded -- skipping.')
else:
    pivot = unet_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='dice')
    if {'LiteUNet', 'AttentionResUNet'} <= set(pivot.columns):
        pivot['headroom_to_attn'] = pivot['AttentionResUNet'] - pivot['LiteUNet']
    display(pivot)

    plot_df = unet_df.copy()
    plot_df['label'] = plot_df['dataset'] + ' / ' + plot_df['class_name']

    fig = px.bar(plot_df, x='label', y='dice', color='model', barmode='group',
                title='U-Net baselines by dataset / class', labels={'label': '', 'dice': 'Test Dice'})
    render(fig, 'unet_baseline_landscape.png')

    # Per-metric small multiples -- Dice alone hides IoU/BIoU/Precision/Sensitivity/HD95
    # differences. Plotly facets make this a one-liner instead of a manual subplot loop.
    metric_cols = ['dice', 'iou', 'biou', 'precision', 'sensitivity', 'hd95']
    available = [m for m in metric_cols if plot_df[m].notna().any()]
    if available:
        long_metrics = plot_df.melt(id_vars=['label', 'model'], value_vars=available,
                                    var_name='metric', value_name='value')
        fig = px.bar(long_metrics, x='label', y='value', color='model', barmode='group',
                    facet_col='metric', facet_col_wrap=3, facet_col_spacing=0.06,
                    title='U-Net baselines -- all metrics, small multiples')
        fig.update_yaxes(matches=None); fig.update_xaxes(tickangle=40)
        render(fig, 'unet_all_metrics_small_multiples.png')

    if unet_csv_frames:
        long_rows = []
        for f in unet_csv_frames:
            for c in [c for c in f['df'].columns if c.startswith('dice_')]:
                for v in f['df'][c].dropna().values:
                    long_rows.append(dict(label=f"{f['dataset']}/{c.replace('dice_', '')}",
                                          model=f['model'], dice=v))
        long_df = pd.DataFrame(long_rows)
        if not long_df.empty:
            long_df['group'] = long_df['label'] + ' — ' + long_df['model']

            # Violin with box + all points overlaid -- distribution shape, not just quartiles.
            fig = px.violin(long_df, x='group', y='dice', color='model', box=True, points='all',
                            title='U-Net per-patient Dice distribution')
            fig.update_xaxes(tickangle=30, title='')
            render(fig, 'unet_per_patient_violin.png')

            # ECDF -- where does one model's distribution actually sit relative to another's?
            fig = px.ecdf(long_df, x='dice', color='group',
                         title='U-Net per-patient Dice -- empirical CDF')
            render(fig, 'unet_per_patient_ecdf.png')


## 4. Discrete vs continuous — absolute Dice, head-to-head

The primary framing for this comparison: **what Dice does each action space actually
deploy**, compared directly against each other -- not against U-Net. A run can have a
small delta-vs-baseline yet still deploy the higher absolute Dice (if its baseline was
already strong), so "which agent wins on delta" and "which agent wins outright" are
genuinely different questions. This section answers the second one; Section 5 answers
the first.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    # Absolute deployed Dice by action space -- no U-Net reference at all.
    fig = px.box(drl_df, x='action_space', y='deploy_dice', color='action_space', points='all',
                hover_data=['dataset', 'class_name', 'model'],
                title='Deployed Dice by action space (absolute -- not vs. U-Net baseline)',
                labels={'deploy_dice': 'Deployed Dice', 'action_space': ''})
    render(fig, 'absolute_dice_by_action_space.png')

    # Sanity check: did one action space simply inherit easier starting masks?
    fig = px.box(drl_df, x='action_space', y='init_dice', color='action_space', points='all',
                hover_data=['dataset', 'class_name', 'model'],
                title='U-Net baseline Dice going INTO each action space (confound check)',
                labels={'init_dice': 'Init (U-Net) Dice', 'action_space': ''})
    render(fig, 'init_dice_by_action_space.png')

    summary_tbl = drl_df.groupby('action_space')['deploy_dice'].agg(
        ['mean', 'median', 'std', 'min', 'max', 'count']).round(4)
    display(summary_tbl)

    # Head-to-head: for every (dataset, class) where BOTH a discrete and a continuous
    # run exist, plot DuelingDDQN's deployed Dice against TD3's -- the cleanest
    # possible "which action space wins on this class" view, with zero U-Net involved.
    pivot_abs = drl_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='deploy_dice')
    discrete_cols = [c for c in pivot_abs.columns if c == 'DuelingDDQN']
    continuous_cols = [c for c in pivot_abs.columns if c == 'TD3']
    if discrete_cols and continuous_cols:
        h2h = pivot_abs[['DuelingDDQN', 'TD3']].dropna().reset_index()
        lims = [h2h[['DuelingDDQN', 'TD3']].min().min() - 0.02, h2h[['DuelingDDQN', 'TD3']].max().max() + 0.02]
        fig = px.scatter(h2h, x='DuelingDDQN', y='TD3', text=h2h['dataset'] + '/' + h2h['class_name'],
                         title='Head-to-head: DuelingDDQN vs TD3 deployed Dice (same class, no baseline involved)')
        fig.add_shape(type='line', x0=lims[0], y0=lims[0], x1=lims[1], y1=lims[1],
                     line=dict(dash='dash', color='gray'))
        fig.update_traces(textposition='top center', marker=dict(size=12))
        fig.update_xaxes(range=lims); fig.update_yaxes(range=lims, scaleanchor='x', scaleratio=1)
        render(fig, 'discrete_vs_continuous_head_to_head.png')
        n_td3_wins = int((h2h['TD3'] > h2h['DuelingDDQN']).sum())
        print(f'Of {len(h2h)} matched (dataset, class) pairs with both agents present: '
              f'TD3 deploys the higher Dice in {n_td3_wins}, DuelingDDQN in {len(h2h) - n_td3_wins}.')
    else:
        print('Need at least one (dataset, class) with BOTH a DuelingDDQN and a TD3 run '
              'to draw the head-to-head scatter -- not enough overlap yet.')


## 5. Discrete vs continuous — delta vs U-Net baseline (supporting view)

A different, still-useful question: **how much learning happened**, i.e. how far each
action space moved the U-Net baseline. `TD3` has no learned STOP action (converge-and-
hold + a `fail_thresh` safety net instead, see `iteris/drl_reeval/re_eval_td3.py`), so
it's structurally exposed to *drift* (deploying past its own best-seen point) in a way
`DuelingDDQN` isn't -- reported explicitly here rather than folded into one number.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    agg = drl_df.groupby('action_space').agg(
        n_runs=('model', 'size'), mean_deploy_delta=('deploy_delta', 'mean'),
        std_deploy_delta=('deploy_delta', 'std'),
        win_rate=('deploy_delta', lambda s: float((s > 0).mean())),
        mean_drift=('drift', 'mean'), mean_final_hd95=('final_hd95', 'mean'),
    ).round(4)
    display(agg)

    fig = make_subplots(rows=1, cols=2, subplot_titles=('Deployable delta-Dice by action space',
                                                         'Drift: best-seen minus deployed Dice'))
    for space in drl_df['action_space'].unique():
        sub = drl_df[drl_df['action_space'] == space]
        fig.add_trace(go.Box(y=sub['deploy_delta'], name=space, legendgroup=space, boxpoints='all'), row=1, col=1)
        fig.add_trace(go.Box(y=sub['drift'], name=space, legendgroup=space, showlegend=False, boxpoints='all'), row=1, col=2)
    fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)
    fig.update_layout(title='Discrete vs continuous -- delta-vs-baseline framing')
    render(fig, 'discrete_vs_continuous_delta.png')

    # Improvement scatter: init Dice (baseline) vs deployed Dice, y=x reference.
    lims = [drl_df[['init_dice', 'deploy_dice']].min().min() - 0.02,
            drl_df[['init_dice', 'deploy_dice']].max().max() + 0.02]
    fig = px.scatter(drl_df, x='init_dice', y='deploy_dice', color='action_space',
                     hover_data=['dataset', 'class_name', 'model'],
                     title='Improvement scatter: U-Net baseline vs deployed Dice')
    fig.add_shape(type='line', x0=lims[0], y0=lims[0], x1=lims[1], y1=lims[1], line=dict(dash='dash', color='gray'))
    fig.update_xaxes(range=lims); fig.update_yaxes(range=lims, scaleanchor='x', scaleratio=1)
    render(fig, 'drl_improvement_scatter.png')

    # Drift scatter: best-seen (GT-privileged ceiling) vs actually-deployed final Dice.
    lims2 = [drl_df[['best_dice_ceiling', 'final_dice']].min().min() - 0.02,
             drl_df[['best_dice_ceiling', 'final_dice']].max().max() + 0.02]
    fig = px.scatter(drl_df, x='best_dice_ceiling', y='final_dice', color='action_space',
                     hover_data=['dataset', 'class_name', 'model'],
                     title='Drift scatter: best-seen ceiling vs deployed final Dice',
                     labels={'best_dice_ceiling': 'Best-seen Dice (GT-privileged ceiling)',
                             'final_dice': 'Deployed final Dice'})
    fig.add_shape(type='line', x0=lims2[0], y0=lims2[0], x1=lims2[1], y1=lims2[1], line=dict(dash='dash', color='gray'))
    fig.update_xaxes(range=lims2); fig.update_yaxes(range=lims2, scaleanchor='x', scaleratio=1)
    render(fig, 'drl_drift_scatter.png')

    # Win / tie / loss stacked bar per agent -- epsilon band around 0 counts as a tie.
    eps = 0.005
    drl_df['outcome'] = drl_df['deploy_delta'].apply(lambda v: 'win' if v > eps else ('loss' if v < -eps else 'tie'))
    counts = drl_df.groupby(['model', 'outcome']).size().reset_index(name='n')
    fig = px.bar(counts, x='model', y='n', color='outcome', barmode='stack',
                category_orders={'outcome': ['win', 'tie', 'loss']},
                color_discrete_map={'win': 'green', 'tie': 'gray', 'loss': 'crimson'},
                title=f'Win / tie / loss per agent (tie band = +/-{eps})', labels={'n': 'Number of runs'})
    render(fig, 'drl_win_tie_loss.png')


## 6. Per-class / per-dataset learning behaviour

Two companion heatmaps: **delta** (positive = genuine learning; near-zero = the
STOP-at-baseline collapse from the project's PBRS fix notes; negative = actively
hurting the baseline) and **absolute deployed Dice** (which is what actually ships,
regardless of how much it moved).

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    heat_delta = drl_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='deploy_delta')
    heat_abs = drl_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='deploy_dice')
    row_labels = [f'{d}/{c}' for d, c in heat_delta.index]

    fig = make_subplots(rows=1, cols=2, subplot_titles=('Delta vs U-Net baseline', 'Absolute deployed Dice'))
    fig.add_trace(go.Heatmap(z=heat_delta.values, x=list(heat_delta.columns), y=row_labels,
                             colorscale='RdYlGn', zmid=0, zmin=-0.05, zmax=0.05,
                             text=np.round(heat_delta.values, 3), texttemplate='%{text}',
                             colorbar=dict(title='delta-Dice', x=0.44)), row=1, col=1)
    fig.add_trace(go.Heatmap(z=heat_abs.values, x=list(heat_abs.columns), y=row_labels,
                             colorscale='Viridis', text=np.round(heat_abs.values, 3), texttemplate='%{text}',
                             colorbar=dict(title='Dice', x=1.0)), row=1, col=2)
    fig.update_layout(title='Learning behaviour: delta vs absolute, side by side', height=350 + 25 * len(row_labels))
    render(fig, 'learning_behaviour_heatmaps.png')

    print('Best-learning (dataset, class) per agent (highest deployable delta-Dice):')
    for m in drl_df['model'].unique():
        sub = drl_df[drl_df['model'] == m].sort_values('deploy_delta', ascending=False)
        if not sub.empty:
            top = sub.iloc[0]
            print(f'  {m:>14s}: {top["dataset"]}/{top["class_name"]}  delta-Dice={top["deploy_delta"]:+.4f}  '
                  f'(absolute Dice={top["deploy_dice"]:.4f})')


## 7. Multi-metric radar comparison

One radar per dataset, one polygon per model, axes = every metric available
(Dice / IoU / BIoU / Precision / Sensitivity, averaged across that dataset's
classes, all absolute values). A single-metric bar chart can make a model look
strong when it's really just winning on Dice while losing on boundary precision
(BIoU) or sensitivity -- the radar shape makes that trade-off visible at a glance.

In [ ]:
radar_metrics = ['dice', 'iou', 'biou', 'precision', 'sensitivity']
have_unet = not unet_df.empty
have_drl_metrics = not drl_df.empty and {'final_iou', 'final_biou', 'final_precision', 'final_sensitivity'} <= set(drl_df.columns)

if not have_unet and not have_drl_metrics:
    print('Need U-Net and/or DRL summaries with IoU/BIoU/Precision/Sensitivity to draw radars -- skipping.')
else:
    datasets = sorted(set(unet_df['dataset'].unique() if have_unet else []) |
                       set(drl_df['dataset'].unique() if have_drl_metrics else []))
    fig = make_subplots(rows=1, cols=len(datasets), specs=[[{'type': 'polar'}] * len(datasets)],
                        subplot_titles=datasets)
    axis_labels = [m.upper() if m in ('iou', 'biou') else m.capitalize() for m in radar_metrics]
    for col, dataset in enumerate(datasets, start=1):
        series = {}
        if have_unet:
            for model, sub in unet_df[unet_df['dataset'] == dataset].groupby('model'):
                series[model] = [float(sub[m].mean()) if sub[m].notna().any() else 0.0 for m in radar_metrics]
        if have_drl_metrics:
            for model, sub in drl_df[drl_df['dataset'] == dataset].groupby('model'):
                series[f'{model} (deploy)'] = [
                    float(sub['deploy_dice'].mean()), float(sub['final_iou'].mean()),
                    float(sub['final_biou'].mean()), float(sub['final_precision'].mean()),
                    float(sub['final_sensitivity'].mean()),
                ]
        for name, vals in series.items():
            fig.add_trace(go.Scatterpolar(r=vals + vals[:1], theta=axis_labels + axis_labels[:1],
                                          name=name, legendgroup=name, showlegend=(col == 1)),
                         row=1, col=col)
    fig.update_polars(radialaxis=dict(range=[0, 1]))
    fig.update_layout(title='Multi-metric radar by dataset', height=500)
    render(fig, 'multi_metric_radar.png')


## 8. Most-improved DRL agent ranking

Two rankings side by side: by **delta vs baseline** (learning behaviour) and by
**absolute deployed Dice** (which agent ships the best mask, independent of how
strong its baseline was) -- see Section 4/5's framing note for why these can
disagree.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    ranked_delta = drl_df.sort_values('deploy_delta', ascending=False)[
        ['dataset', 'class_name', 'model', 'action_space', 'init_dice', 'deploy_dice',
         'deploy_delta', 'best_dice_ceiling', 'drift', 'refinable_frac']
    ].reset_index(drop=True)
    print('-- Ranked by delta vs U-Net baseline --')
    display(ranked_delta)

    ranked_abs = drl_df.sort_values('deploy_dice', ascending=False)[
        ['dataset', 'class_name', 'model', 'action_space', 'deploy_dice', 'init_dice', 'deploy_delta']
    ].reset_index(drop=True)
    print('-- Ranked by absolute deployed Dice --')
    display(ranked_abs)

    per_agent = drl_df.groupby('model').agg(
        n_runs=('deploy_delta', 'size'), mean_deploy_delta=('deploy_delta', 'mean'),
        mean_deploy_dice=('deploy_dice', 'mean'), median_deploy_dice=('deploy_dice', 'median'),
        n_positive=('deploy_delta', lambda s: int((s > 0).sum())),
    ).round(4)
    display(per_agent.sort_values('mean_deploy_delta', ascending=False))

    winner_delta = per_agent['mean_deploy_delta'].idxmax()
    winner_abs = per_agent['mean_deploy_dice'].idxmax()
    print()
    print(f'>>> Most improved (mean delta-Dice): {winner_delta} '
          f'({per_agent.loc[winner_delta, "mean_deploy_delta"]:+.4f})')
    print(f'>>> Highest absolute deployed Dice:  {winner_abs} '
          f'({per_agent.loc[winner_abs, "mean_deploy_dice"]:.4f})'
          + ('  -- same agent wins both' if winner_delta == winner_abs else '  -- these DISAGREE, see framing note above'))

    # Before/after dumbbell -- every run as a line from init Dice to deployed Dice.
    plot_rows = drl_df.sort_values('deploy_dice').reset_index(drop=True)
    ylabels = [f"{r['dataset']}/{r['class_name']} — {r['model']}" for _, r in plot_rows.iterrows()]
    fig = go.Figure()
    for i, r in plot_rows.iterrows():
        color = 'green' if r['deploy_dice'] >= r['init_dice'] else 'crimson'
        fig.add_trace(go.Scatter(x=[r['init_dice'], r['deploy_dice']], y=[ylabels[i]] * 2,
                                 mode='lines', line=dict(color=color, width=2), showlegend=False))
    fig.add_trace(go.Scatter(x=plot_rows['init_dice'], y=ylabels, mode='markers',
                             marker=dict(color='gray', size=9), name='U-Net baseline'))
    fig.add_trace(go.Scatter(x=plot_rows['deploy_dice'], y=ylabels, mode='markers',
                             marker=dict(color=['green' if d >= i else 'crimson'
                                                for d, i in zip(plot_rows['deploy_dice'], plot_rows['init_dice'])],
                                        size=9), name='deployed'))
    fig.update_layout(title='Before (baseline) -> after (deployed) Dice, every run',
                      xaxis_title='Dice', height=max(400, 28 * len(plot_rows)))
    render(fig, 'drl_before_after_dumbbell.png')


## 9. Qualitative visual gallery

Every comparison/behaviour/learning-curve/qualitative-grid image found, displayed
inline. These are pre-rendered PNGs the training pipeline already produced, so this
is a raster viewer rather than a chart -- shown via matplotlib for that reason (no
data to re-visualize with Plotly here). This is the "does it actually look right"
check that aggregate Dice numbers can hide.

In [ ]:
def _show_gallery(entries, ncols=2, figsize_per=(6, 5)):
    if not entries:
        return
    n = len(entries)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0]*ncols, figsize_per[1]*nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, entry in zip(axes, entries):
        try:
            img = mpimg.imread(entry['path'])
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f'could not load:\n{e}', ha='center', va='center', wrap=True)
        ax.set_title(entry['caption'], fontsize=9)
        ax.axis('off')
    for ax in axes[len(entries):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

for key, title in [('comparison', 'DRL init vs refined vs GT (comparison plots)'),
                    ('behaviour', 'DRL per-episode Dice trajectories (behaviour plots)'),
                    ('learning_curves', 'U-Net training curves'),
                    ('qualitative', 'U-Net qualitative prediction grids')]:
    entries = png_gallery.get(key, [])
    print(f'--- {title}: {len(entries)} image(s) ---')
    _show_gallery(entries)

if png_gallery.get('other'):
    print(f"{len(png_gallery['other'])} other PNG(s) found (not matching a known naming pattern):")
    for e in png_gallery['other']:
        print(f'    - {e["path"]}')


## 10. BRISC tumor-type classifier evaluation

The classifier (`iteris.classifier.TumorClassifier`) predicts tumor type
(glioma / meningioma / pituitary / non_tumor) — a separate task from the
segmentation models above, evaluated on accuracy + per-class precision/recall/F1
rather than Dice.

In [ ]:
if not classifier_summaries:
    print("""
No brisc_tumor_classifier_summary.json found -- skipping. Run the eval cell in the
classifier notebook (loads the trained checkpoint, calls
iteris.classifier.evaluate_classifier_test_set, then save_classifier_summary_json)
and drop its output under a searched root.
""")
else:
    for summ in classifier_summaries:
        print(f"Source: {summ['source']}")
        print(f"  Best val accuracy: {summ['best_val_acc']}   "
              f"Test accuracy: {summ['accuracy']}  (n={summ['n_test']})   "
              f"Epochs trained: {summ['epochs_trained']}")
        pc = pd.DataFrame(summ['per_class']).T
        display(pc)

        long_pc = pc.reset_index().rename(columns={'index': 'class'}).melt(
            id_vars='class', value_vars=['precision', 'recall', 'f1'], var_name='metric', value_name='score')
        fig = px.bar(long_pc, x='class', y='score', color='metric', barmode='group',
                    title='Per-class precision / recall / F1', range_y=[0, 1.05])
        render(fig, 'classifier_per_class_scores.png')

        # Radar view of the same numbers -- makes an uneven class visually obvious.
        fig = go.Figure()
        cats = ['Precision', 'Recall', 'F1']
        for cls, row in pc.iterrows():
            vals = [row['precision'], row['recall'], row['f1']]
            fig.add_trace(go.Scatterpolar(r=vals + vals[:1], theta=cats + cats[:1], name=cls))
        fig.update_polars(radialaxis=dict(range=[0, 1]))
        fig.update_layout(title='Classifier per-class radar')
        render(fig, 'classifier_radar.png')

        # Native gauge charts -- overall accuracy / macro-F1 at a glance.
        macro_f1 = float(pc['f1'].mean())
        fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'indicator'}, {'type': 'indicator'}]])
        fig.add_trace(go.Indicator(mode='gauge+number', value=summ['accuracy'], title={'text': 'Accuracy'},
                                   gauge={'axis': {'range': [0, 1]}}), row=1, col=1)
        fig.add_trace(go.Indicator(mode='gauge+number', value=macro_f1, title={'text': 'Macro F1'},
                                   gauge={'axis': {'range': [0, 1]}}), row=1, col=2)
        fig.update_layout(title='Overall classifier gauge')
        render(fig, 'classifier_gauge.png')

    print(f"--- Confusion matrix image(s): {len(png_gallery['classifier_confusion'])} ---")
    _show_gallery(png_gallery['classifier_confusion'], ncols=1, figsize_per=(6, 6))
    print(f"--- Learning curve image(s): {len(png_gallery['classifier_curves'])} ---")
    _show_gallery(png_gallery['classifier_curves'], ncols=1, figsize_per=(9, 4))


## 11. Checkpoint / training provenance

Scalar metadata only (step / epoch / best_dice / val_acc), read straight out of the
`.pt` files with plain `torch.load` — never rebuilds a model, so this works without
`iteris` or `monai` installed. Useful for sanity-checking which checkpoint got
deployed (`*_best.pt`) vs how far training actually ran (`*_stepN.pt` milestones).

In [ ]:
if not HAVE_TORCH:
    print('torch is not importable in this environment -- skipping (this section is optional).')
elif ckpt_df.empty:
    print('No .pt/.pth checkpoints found under the searched roots.')
else:
    display(ckpt_df.sort_values('path'))


## 12. Statistical significance (Wilcoxon signed-rank + Bonferroni)

Two sources of real paired data are used, both tested here with the same
Bonferroni correction applied jointly across every test run in this section:

1. **U-Net per-patient CSVs** (`*_test_scores.csv`) — if two U-Net models were
   evaluated on the same dataset (e.g. LiteUNet vs AttentionResUNet), their
   per-patient Dice is paired by `patient` ID and tested per class. This data
   already exists automatically from `evaluate_test_set` — nothing extra to export.
2. **Generic paired CSVs** (`init_dice`/`final_dice` columns) — for DRL runs, if
   you exported per-sample data.

Aggregate JSON summaries alone (Sections 1-8) are already means and can't support
a real hypothesis test on their own.

In [ ]:
sig_rows = []

# (1) U-Net per-patient, paired across model pairs sharing a dataset.
by_dataset = {}
for f in unet_csv_frames:
    by_dataset.setdefault(f['dataset'], []).append(f)
for dataset, frames in by_dataset.items():
    for i in range(len(frames)):
        for j in range(i + 1, len(frames)):
            a, b = frames[i], frames[j]
            if a['model'] == b['model']:
                continue
            merged = a['df'].merge(b['df'], on='patient', suffixes=('_a', '_b'))
            dice_cols = [c for c in a['df'].columns if c.startswith('dice_')]
            for c in dice_cols:
                ca, cb = f'{c}_a', f'{c}_b'
                if ca not in merged.columns or cb not in merged.columns:
                    continue
                paired = merged[[ca, cb]].dropna()
                if len(paired) < 5:
                    continue
                stat, p = stats.wilcoxon(paired[cb], paired[ca])
                sig_rows.append(dict(
                    comparison=f"{dataset}/{c.replace('dice_', '')}: {b['model']} vs {a['model']}",
                    n=len(paired), mean_delta=(paired[cb] - paired[ca]).mean(),
                    wilcoxon_stat=stat, p_raw=p))

# (2) Generic paired CSVs (DRL per-sample, if ever exported).
for path, df in generic_paired_frames.items():
    paired = df[['init_dice', 'final_dice']].dropna()
    if len(paired) < 5:
        print(f'  [skip] {path}: only {len(paired)} paired samples, too few for Wilcoxon')
        continue
    stat, p = stats.wilcoxon(paired['final_dice'], paired['init_dice'])
    sig_rows.append(dict(comparison=path, n=len(paired),
                          mean_delta=(paired['final_dice'] - paired['init_dice']).mean(),
                          wilcoxon_stat=stat, p_raw=p))

if sig_rows:
    sig_df = pd.DataFrame(sig_rows)
    sig_df['p_bonferroni'] = np.minimum(sig_df['p_raw'] * len(sig_df), 1.0)
    sig_df['significant_at_0.05'] = sig_df['p_bonferroni'] < 0.05
    display(sig_df)
else:
    print("""No paired data available for significance testing yet.
Either: (a) no two U-Net models share a dataset with matching per-patient CSVs, or
(b) no *_test_scores.csv / generic paired CSVs were found at all. Drop the
*_test_scores.csv files evaluate_test_set already produces for each U-Net run,
and/or per-sample DRL exports, then re-run.""")


### 12a. DRL metric correlation

How the DRL metrics relate to each other across every loaded run -- e.g. does a
larger deployable delta-Dice actually correlate with better boundary precision
(delta-BIoU), or can a run "win" on Dice while trading away boundary quality?
Also shows whether `refinable_frac` predicts the size of the deployable gain.

In [ ]:
corr_cols = ['deploy_delta', 'deploy_dice', 'delta_iou', 'delta_biou', 'drift', 'refinable_frac', 'init_dice']
corr_cols = [c for c in corr_cols if c in drl_df.columns and drl_df[c].notna().sum() >= 3]
if drl_df.empty or len(corr_cols) < 2:
    print('Not enough DRL runs/metrics loaded to compute a correlation matrix -- skipping.')
else:
    corr = drl_df[corr_cols].corr()
    fig = px.imshow(corr, x=corr_cols, y=corr_cols, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                    text_auto='.2f', title='DRL metric correlation (across all loaded runs)')
    render(fig, 'drl_metric_correlation.png')


## 13. Export & auto-generated summary

Saves the master table and prints a summary built entirely from computed values —
nothing here is pre-written prose.

**Context for interpreting a "DRL didn't beat baseline" result** (from this project's
own diagnostics, `docs/CONTEXT.md`): three separate, non-overlapping caps can each
produce that outcome, and only the last is an optimization problem the training
recipe can fix -- (a) **representation cap**: the deployed mask is a smoothed
periodic-spline approximation of the largest connected component, never the raw
U-Net mask; (b) **information cap**: a GT-blind refiner has no privileged signal the
U-Net lacked; (c) **optimization cap**: reward shaping / STOP-timing / training
budget. Run `iteris.diagnostics.headroom_report` to tell these apart for any run
showing near-zero or negative delta-Dice.

In [ ]:
if not master_df.empty:
    out_csv = OUT_DIR / 'master_comparison.csv'
    master_df.to_csv(out_csv, index=False)
    print(f'[export] master comparison table -> {out_csv}')

print()
print('=' * 70)
print('SUMMARY (auto-generated from loaded results)')
print('=' * 70)

if unet_df.empty and drl_df.empty and not classifier_summaries:
    print('No result files loaded yet -- nothing to summarize. See Section 1.')
else:
    if not unet_df.empty:
        best_unet = unet_df.sort_values('dice', ascending=False).iloc[0]
        print(f'- Strongest U-Net baseline observed: {best_unet["model"]} on '
              f'{best_unet["dataset"]}/{best_unet["class_name"]} (Dice={best_unet["dice"]:.4f}).')
    if not drl_df.empty:
        abs_by_space = drl_df.groupby('action_space')['deploy_dice'].mean()
        for space, val in abs_by_space.items():
            print(f'- Mean ABSOLUTE deployed Dice, {space} action space: {val:.4f}')
        if len(abs_by_space) >= 2:
            print(f'  -> {abs_by_space.idxmax()} action space deploys the higher Dice outright '
                  f'(not vs. baseline).')
        delta_by_space = drl_df.groupby('action_space')['deploy_delta'].mean()
        for space, val in delta_by_space.items():
            print(f'- Mean delta-Dice vs baseline, {space} action space: {val:+.4f}')
        per_agent_abs = drl_df.groupby('model')['deploy_dice'].mean().sort_values(ascending=False)
        print(f'- Highest absolute deployed Dice overall: {per_agent_abs.index[0]} '
              f'(mean Dice={per_agent_abs.iloc[0]:.4f}).')
        per_agent_delta = drl_df.groupby('model')['deploy_delta'].mean().sort_values(ascending=False)
        print(f'- Most improved overall (mean delta-Dice): {per_agent_delta.index[0]} '
              f'(mean delta-Dice={per_agent_delta.iloc[0]:+.4f}).')
        n_beat = int((drl_df['deploy_delta'] > 0).sum())
        print(f'- {n_beat}/{len(drl_df)} loaded DRL runs beat their U-Net baseline (deployable comparison).')
    if classifier_summaries:
        accs = [s['accuracy'] for s in classifier_summaries if s['accuracy'] is not None]
        if accs:
            print(f'- BRISC tumor-type classifier test accuracy: {accs[0]:.4f}'
                  + (f' (mean over {len(accs)} runs)' if len(accs) > 1 else '') + '.')
    n_gallery = sum(len(v) for v in png_gallery.values())
    print(f'- Qualitative gallery: {n_gallery} image(s) displayed in Section 9/10.')
    if HAVE_TORCH and not ckpt_df.empty:
        print(f'- Checkpoint provenance: {len(ckpt_df)} .pt file(s) inspected in Section 11.')
    n_figs_saved = len(list(FIG_DIR.glob('*.png')))
    if HAVE_KALEIDO and not _kaleido_broken:
        print(f'- {n_figs_saved} chart(s) rendered as static PNGs (visible on GitHub/nbviewer too), '
              f'also saved to {FIG_DIR.resolve()}.')
    else:
        print('- kaleido unavailable/broken in this environment -- charts used the interactive '
              'fallback renderer, which will NOT display on GitHub/nbviewer (only in a live '
              'Jupyter/Kaggle session). See the note(s) above for why.')
print('=' * 70)
